# 09 — Prepare NF-ToN-IoT-v2 and NF-BoT-IoT-v2

Adds two corpora to the existing NF-CSE-CIC-IDS2018-v2 and NF-UNSW-NB15-v2 caches, giving four corpora on the identical 43-feature NetFlow v2 schema. NF-BoT-IoT-v2 (37.8M flows) is streamed in batches through pyarrow and never fully materialised. Both new corpora are proportionally subsampled to 3M flows with native prevalence preserved, matching the treatment of NF-CSE-CIC-IDS2018-v2. Schema equality across all four caches is asserted, and a corpora manifest with exact counts is written for the manuscript.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, gc, json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE   = '/content/drive/MyDrive/drift-conference'
DATA   = f'{BASE}/data/nfv2'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/nfv2'
for d in (DATA, CACHE, RESULT):
    os.makedirs(d, exist_ok=True)

CFG = dict(seed=42, n_max=3_000_000, oversample=1.15, batch_rows=1_000_000)

KAGGLE = {
    'nftonv2': 'dhoogla/nftoniotv2',
    'nfbotv2': 'dhoogla/nfbotiotv2',
}
IDENTIFIERS = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Dataset']
print(json.dumps(CFG, indent=2))

In [ ]:
def have_parquet(tag):
    return len(glob.glob(f'{DATA}/{tag}/*.parquet')) > 0

need = [t for t in KAGGLE if not have_parquet(t)]
if need:
    !pip install -q kaggle
    os.makedirs('/root/.kaggle', exist_ok=True)
    assert os.path.exists('/content/drive/MyDrive/kaggle.json'), 'kaggle.json missing at MyDrive/'
    !cp /content/drive/MyDrive/kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json
    for tag in need:
        os.makedirs(f'{DATA}/{tag}', exist_ok=True)
        !kaggle datasets download -d {KAGGLE[tag]} -p {DATA}/{tag} --unzip
for tag in KAGGLE:
    files = sorted(glob.glob(f'{DATA}/{tag}/*.parquet'))
    print(tag, '->', [os.path.basename(f) for f in files])
    assert files, f'no parquet for {tag}'

In [ ]:
def stream_sample(paths, n_max, oversample, batch_rows, seed):
    """Uniform row sampling across parquet batches without materialising the file.
    Returns (sampled_df, total_rows). Oversamples slightly; exact trim follows."""
    total = sum(pq.ParquetFile(p).metadata.num_rows for p in paths)
    p_keep = min(1.0, oversample * n_max / total)
    rng = np.random.default_rng(seed)
    parts = []
    for path in paths:
        pf = pq.ParquetFile(path)
        for batch in pf.iter_batches(batch_size=batch_rows):
            df = batch.to_pandas()
            keep = rng.random(len(df)) < p_keep
            parts.append(df[keep])
            del df
        gc.collect()
    return pd.concat(parts, ignore_index=True), total

def prepare(df, n_max, seed):
    df = df.drop(columns=[c for c in IDENTIFIERS if c in df.columns])
    if len(df) > n_max:
        frac = n_max / len(df)
        df = df.groupby('Attack', group_keys=False).sample(frac=frac, random_state=seed)
    return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

manifest = {}
for tag in KAGGLE:
    cache = f'{CACHE}/{tag}_prepared.parquet'
    if os.path.exists(cache):
        d = pd.read_parquet(cache)
        total = pq.ParquetFile(sorted(glob.glob(f'{DATA}/{tag}/*.parquet'))[0]).metadata.num_rows
        print(f'{tag}: cache present ({len(d):,} rows)')
    else:
        raw, total = stream_sample(sorted(glob.glob(f'{DATA}/{tag}/*.parquet')),
                                   CFG['n_max'], CFG['oversample'], CFG['batch_rows'], CFG['seed'])
        print(f'{tag}: streamed {len(raw):,} of {total:,} rows')
        d = prepare(raw, CFG['n_max'], CFG['seed'])
        del raw
        gc.collect()
        d.to_parquet(cache, index=False)
        print(f'{tag}: cached {len(d):,} rows')
    fam = d['Attack'].value_counts()
    manifest[tag] = dict(total_rows=int(total), prepared_rows=int(len(d)),
                         attack_rate=round(float(d['Label'].mean()), 4),
                         n_attack_families=int((fam.index != 'Benign').sum()),
                         families={k: int(v) for k, v in fam.items()})
    print(f"  attack rate {manifest[tag]['attack_rate']}, families {manifest[tag]['n_attack_families']}")
    del d
    gc.collect()

In [ ]:
# Schema equality across all four prepared caches; manifest for the existing two.
ALL = ['nf2018v2', 'nfunswv2', 'nftonv2', 'nfbotv2']
TOTALS = {'nf2018v2': 17_129_715, 'nfunswv2': 1_986_745}
feature_sets = {}
for tag in ALL:
    d = pd.read_parquet(f'{CACHE}/{tag}_prepared.parquet')
    feats = [c for c in d.columns if c not in ('Label', 'Attack')]
    feature_sets[tag] = feats
    if tag not in manifest:
        fam = d['Attack'].value_counts()
        manifest[tag] = dict(total_rows=TOTALS[tag], prepared_rows=int(len(d)),
                             attack_rate=round(float(d['Label'].mean()), 4),
                             n_attack_families=int((fam.index != 'Benign').sum()),
                             families={k: int(v) for k, v in fam.items()})
    print(f'{tag}: {len(d):,} rows, {len(feats)} features, attack rate {d.Label.mean():.4f}')
    del d
    gc.collect()

ref = feature_sets['nf2018v2']
for tag, feats in feature_sets.items():
    assert feats == ref, f'schema mismatch for {tag}: {set(feats) ^ set(ref)}'
print(f'\nschema identical across all four corpora: {len(ref)} features')

with open(f'{RESULT}/corpora_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print('manifest written')
for tag in ALL:
    m = manifest[tag]
    print(f"{tag:10s} total {m['total_rows']:>12,}  prepared {m['prepared_rows']:>10,}  "
          f"attack {m['attack_rate']:.4f}  families {m['n_attack_families']}")

In [ ]:
subprocess.run(["python", "tools/commit_cell.py",
  "09: prepare NF-ToN-IoT-v2 and NF-BoT-IoT-v2 (streamed, 3M proportional); four-corpus schema verified; corpora manifest; add tools/commit_cell.py"])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "09: prepare NF-ToN-IoT-v2 and NF-BoT-IoT-v2 (streamed, 3M proportional); four-corpus schema verified; corpora manifest; add tools/commit_cell.py"],
  capture_output=True, text=True)
print(r.stdout)
print(r.stderr)